In [ ]:
import torch
import numpy as np
import random
import os
import math

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True  # Enforce deterministic algorithms
        torch.backends.cudnn.benchmark = False     # Disable benchmark for reproducibility

    os.environ['PYTHONHASHSEED'] = str(seed)       # Seed Python hashing, which can affect ordering
set_seed(42)

### Explicit Heuristic Split Model

#### Segmentation Pipeline GroundingDINO+SAM through Autodistill GroundingSAM

In [ ]:
import os
import h5py
import numpy as np
from autodistill_grounded_sam import GroundedSAM
from autodistill.detection import CaptionOntology
from tqdm.notebook import tqdm
import supervision as sv
import random

In [ ]:
# Initialize model with ontology
# define an ontology to map class names to our GroundedSAM prompt
# the ontology dictionary has the format {caption: class}
# where caption is the prompt sent to the base model, and class is the label that will
# be saved for that caption in the generated annotations
# then, load the model
base_model = GroundedSAM(
    ontology=CaptionOntology(
        {
            "human . child . person": "human",
            "robot": "robot",
            "dog": "dog"
        }
    )
)

#### Helpers

In [ ]:
# Define categories for images
def get_category(image_name: str) -> str:
    categories = ["Hallway", "BigOffice-2", "BigOffice-3", "MeetingRoom", "SmallOffice", "Home"]
    for cat in categories:
        if cat in image_name:
            return cat
    return "Home"

# Helper to filter Detections object dict with only needed attributes
def filter_detections(detections):
    filtered = {
        "xyxy": detections.xyxy,
        "mask": detections.mask,
        "confidence": detections.confidence,
        "class_id": detections.class_id,
        # discard the rest
    }
    return filtered

def load_detection_from_h5(img_group) -> sv.Detections:
    xyxy = img_group["xyxy"][:]            # numpy array (N,4)
    masks = img_group["mask"][:]            # numpy array (N, H, W)
    confidence = img_group["confidence"][:] # numpy array (N,)
    class_id = img_group["class_id"][:]     # numpy array (N,)

    detection = sv.Detections(
        xyxy=xyxy,
        mask=masks,
        confidence=confidence,
        class_id=class_id,
    )
    return detection


#### Panoptic segmentation

In [ ]:
def segment_images(
    input_image_dir="../data/images/",
    image_extension=".png",
    output_hdf5_path="autodistill_segmentation_dataset.hdf5",
    batch_size=20,
    ) -> None:
    """Perform panoptic segmentation and save the results to a HDF5 dataset."""

    all_images = [f for f in os.listdir(input_image_dir) if f.endswith(image_extension)]

    # Create HDF5 file
    if os.path.exists(output_hdf5_path):
        raise FileExistsError(f"Filec '{output_hdf5_path}' already exists.")
    else:
        h5file = h5py.File(output_hdf5_path, "a")

    # Process images
    for i in tqdm(range(0, len(all_images), batch_size), desc="Batch"):
        batch = all_images[i : i + batch_size]
        for image_name in tqdm(batch, desc="Image", leave=False):
            img_path = os.path.join(input_image_dir, image_name)
            
            detections = base_model.predict(img_path)
            data = filter_detections(detections)

            # Create group for category
            category = get_category(image_name)
            if category not in h5file:
                h5file.create_group(category)
            category_group = h5file[category]

            # Create subgroup for image name
            if image_name in category_group:
                raise RuntimeError(f"Duplicate entry detected for image '{image_name}' in category '{category}'.")
            img_group = category_group.create_group(image_name)

            # Save detection data to datasets in img_group
            img_group.create_dataset("xyxy", data=data["xyxy"], compression="gzip")
            img_group.create_dataset("mask", data=data["mask"], compression="gzip")
            img_group.create_dataset("confidence", data=data["confidence"], compression="gzip")
            img_group.create_dataset("class_id", data=data["class_id"], compression="gzip")

        # Flush to disk after batch
        h5file.flush()
    h5file.close()

In [ ]:
# segment_images(image_dir="../data/images/", image_extension=".png", hdf5_path="autodistill_segmentation_dataset.hdf5", batch_size=20)

#### Utils for inspecting and visualising masks

In [ ]:
def inspect_detections_dataset(hdf5_path = "autodistill_segmentation_dataset.hdf5"):
    with h5py.File(hdf5_path, "r") as h5file:
        categories = list(h5file.keys())
        print(f"Categories ({len(categories)}): {categories}\n")

        total_images = 0
        category_counts = {}
        image_paths = []

        for category in categories:
            images = list(h5file[category].keys())
            count = len(images)
            category_counts[category] = count
            total_images += count
            # Save image full path as (category, image_name)
            image_paths.extend([(category, img) for img in images])

        print("Number of images per category:")
        for cat, cnt in category_counts.items():
            print(f"  {cat}: {cnt}")
        print(f"\nTotal images: {total_images}\n")

        # Sample 20 random images for inspection
        sample_paths = random.sample(image_paths, min(20, len(image_paths)))
        print("Sample 20 images (category, image_name) and dataset shapes:")

        for category, image_name in sample_paths:
            img_group = h5file[category][image_name]
            print(f"\n{category}/{image_name}:")
            for dataset_name in img_group.keys():
                data = img_group[dataset_name]
                print(f"  - {dataset_name}: shape={data.shape}, dtype={data.dtype}")

In [ ]:
# inspect_detections_dataset(hdf5_path="autodistill_segmentation_dataset.hdf5")

In [ ]:
import os
import h5py
import numpy as np
import cv2
from PIL import Image, ImageDraw
from tqdm.notebook import tqdm

def apply_bbox_mask_to_dataset(
        image_dir="../data/images/",
        hdf5_path="autodistill_segmentation_dataset.hdf5",
        output_dir="../data/test_segmentations_colored/",
        confidence_threshold=0.3,
        resize_to=(512, 288),
        deduplicate_bbox=False
        ):

    # Define class colors with alpha (RGBA)
    class_colors = {
        0: (255, 0, 0, 100),    # Red (human)
        1: (0, 255, 0, 100),    # Green (robot)
        2: (0, 0, 255, 100),    # Blue (animal)
        3: (122, 122, 0, 100),
        # Add more classes/colors as needed
    }

    
    os.makedirs(output_dir, exist_ok=True)
    target_width, target_height = resize_to
    

    with h5py.File(hdf5_path, "r") as h5file:
        for category in tqdm(h5file.keys(), desc="Categories"):
            category_group = h5file[category]
            for image_name in tqdm(category_group.keys(), desc=f"Images in {category}", leave=False):
                img_group = category_group[image_name]
                detections = load_detection_from_h5(img_group)
                if deduplicate_bbox:
                    detections = detections.with_nms(class_agnostic=True)

                xyxy = detections.xyxy
                masks = detections.mask
                confidences = detections.confidence
                class_ids = detections.class_id

                img_path = os.path.join(image_dir, image_name)
                image = cv2.imread(img_path)
                if image is None:
                    print(f"Warning: Image {image_name} not found, skipping.")
                    continue

                orig_h, orig_w = image.shape[:2]
                scale_x = target_width / orig_w
                scale_y = target_height / orig_h

                rgb_image = image[:, :, ::-1]
                pil_img = Image.fromarray(rgb_image).convert("RGBA")
                pil_img = pil_img.resize((target_width, target_height), resample=Image.LANCZOS)

                # Filter detections by confidence threshold
                keep = confidences >= confidence_threshold
                if not np.any(keep):
                    print(f"No detections above threshold for {image_name}")
                    continue

                filtered_boxes = xyxy[keep]
                filtered_masks = masks[keep]
                filtered_confs = confidences[keep]
                filtered_classes = class_ids[keep]

                # Scale bounding boxes to resized image
                scaled_boxes = []
                for box in filtered_boxes:
                    x1, y1, x2, y2 = box
                    scaled_box = (
                        int(x1 * scale_x),
                        int(y1 * scale_y),
                        int(x2 * scale_x),
                        int(y2 * scale_y),
                    )
                    scaled_boxes.append(scaled_box)

                # Resize masks to target size (nearest neighbor)
                scaled_masks = []
                for mask in filtered_masks:
                    pil_mask = Image.fromarray((mask * 255).astype(np.uint8))
                    pil_mask = pil_mask.resize((target_width, target_height), resample=Image.NEAREST)
                    scaled_mask = np.array(pil_mask) > 128 #turn to binary
                    scaled_masks.append(scaled_mask)

                # Overlay colored masks with transparency
                for mask, class_id in zip(scaled_masks, filtered_classes):
                    color = class_colors.get(class_id, (255, 255, 255, 100))  # default white
                    mask_img = Image.fromarray((mask.astype(bool) * 255).astype(np.uint8), mode="L")
                    colored_mask = Image.new("RGBA", pil_img.size, color)
                    pil_img = Image.alpha_composite(pil_img, Image.composite(colored_mask, Image.new("RGBA", pil_img.size), mask_img))

                # Draw bounding boxes and confidence text
                draw = ImageDraw.Draw(pil_img)
                for (x1, y1, x2, y2), conf, class_id in zip(scaled_boxes, filtered_confs, filtered_classes):
                    color_rgb = class_colors.get(class_id, (255, 255, 255, 100))[:3]
                    draw.rectangle([x1, y1, x2, y2], outline=color_rgb, width=2)
                    draw.text((x1, max(y1 - 10 * (class_id + 1), 0)), f"{class_id}: {conf:.2f}", fill=color_rgb)

                # Save output image as PNG
                save_path = os.path.join(output_dir, os.path.splitext(image_name)[0] + ".png")
                pil_img.convert("RGB").save(save_path)

    print("All images processed and saved with resized overlays.")


In [ ]:
def get_dataset(hdf5_path="autodistill_segmentation_dataset.hdf5", image_dir="../data/images/", category_name="Hallway", deduplicate_bbox=False):
    """
    Get the dataset for image decomposition. Returns a list[tuple] of image paths with segmentation object (detections).
    """
    detections_dataset: list[tuple] = []

    with h5py.File(hdf5_path, "r") as h5file:
        category_group = h5file[category_name]
        for image_name in category_group.keys():
            img_group = category_group[image_name]
            detections = load_detection_from_h5(img_group)
            if deduplicate_bbox:
                detections = detections.with_nms(class_agnostic=True)
            img_path = os.path.join(image_dir, image_name)
            detections_dataset.append((img_path, detections))

    print(f"Loaded {len(detections_dataset)} images and detections from category '{category_name}'.")
    return detections_dataset


#### Image Decomposition -> Soc + Env

In [ ]:
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
import os
from pathlib import Path

def process_segmentation_data(detections_dataset, output_dir, imagenet_mean=[0.485, 0.456, 0.406], confidence_threshold=0.3):
    """
    Process supervision Detections dataset to create social and environment images
    
    Args:
        detections_dataset: list[tuple] (image_path, Detection_object, ...)
        output_dir: Base directory to save processed images
        imagenet_mean: RGB mean values for filling masked areas
    """
    
    # Create output directories
    social_dir = Path(output_dir) / "social"
    env_dir = Path(output_dir) / "environment"
    social_dir.mkdir(parents=True, exist_ok=True)
    env_dir.mkdir(parents=True, exist_ok=True)
    
    for item in tqdm(detections_dataset):
        image_path = item[0]
        detections = item[-1]
        
        # Load original image
        original_image = cv2.imread(str(image_path))
        original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
        height, width = original_image.shape[:2]
        
        # Get filename for saving
        filename = Path(image_path).name

        keep_indices = [i for i, conf in enumerate(detections.confidence) if conf >= confidence_threshold]
        filtered_masks = detections.mask[keep_indices]
        
        # Combine all masks into one
        combined_mask = combine_masks(filtered_masks, height, width)
        
        # Create social image (only people and robot visible)
        social_image = apply_mask_with_mean(
            original_image, combined_mask, imagenet_mean, keep_masked=True
        )
        
        # Create environment image (room only, people and robot masked out)
        env_image = apply_mask_with_mean(
            original_image, combined_mask, imagenet_mean, keep_masked=False
        )
        
        # Save images
        social_path = social_dir / filename
        env_path = env_dir / filename
        
        save_image(social_image, social_path)
        save_image(env_image, env_path)


def combine_masks(masks, height, width, confidence_threshold=0.3):
    """
    Combine multiple masks into a single binary mask using union operation
    
    Args:
        masks: Array of individual masks from Detection object
        height, width: Dimensions of the original image
        
    Returns:
        combined_mask: Single binary mask (1 = object, 0 = background)
    """
    if masks is None or len(masks) == 0:
        return np.zeros((height, width), dtype=np.uint8)
    
    # Initialize combined mask
    combined_mask = np.zeros((height, width), dtype=np.uint8)
    
    # Union all individual masks using maximum function (as shown in search results)
    for mask in masks:
        # Ensure mask is the right size
        if mask.shape != (height, width):
            raise ValueError(f"Mask shape incorrect: {mask.shape}, should be {(height, width)}")
        
        # Union operation: take maximum of current combined mask and new mask
        combined_mask = np.maximum(combined_mask, mask.astype(np.uint8))
    
    return combined_mask

def apply_mask_with_mean(image, mask, imagenet_mean, keep_masked=True):
    """
    Apply mask to image and fill empty areas with ImageNet mean values
    
    Args:
        image: Original RGB image (H, W, 3)
        mask: Binary mask (H, W) where 1 = object, 0 = background
        imagenet_mean: RGB mean values [R, G, B] in range [0, 1]
        keep_masked: If True, keep masked areas (social). If False, remove masked areas (environment)
        
    Returns:
        processed_image: Image with mask applied and filled with mean values
    """
    processed_image = image.copy().astype(np.float32) / 255.0
    
    # Convert imagenet_mean to same range as image
    mean_values = np.array(imagenet_mean).reshape(1, 1, 3)
    
    if keep_masked:
        # Social image: keep people/robot, fill background with mean
        fill_mask = (mask == 0)  # Areas to fill (background)
    else:
        # Environment image: keep background, fill people/robot with mean  
        fill_mask = (mask == 1)  # Areas to fill (people/robot)
    
    # Fill specified areas with ImageNet mean values
    for c in range(3):  # RGB channels
        processed_image[:, :, c][fill_mask] = imagenet_mean[c]
    
    # Convert back to uint8
    processed_image = (processed_image * 255).astype(np.uint8)
    
    return processed_image

def save_image(image, save_path):
    """
    Save image to specified path
    
    Args:
        image: RGB image array (H, W, 3)
        save_path: Path to save the image
    """
    # Convert to PIL Image and save
    pil_image = Image.fromarray(image)
    pil_image.save(save_path)


In [ ]:
# dataset_home = get_dataset(
#     hdf5_path="autodistill_segmentation_dataset.hdf5", 
#     image_dir="../data/images/", 
#     category_name="Home", 
#     deduplicate_bbox=False)

# process_segmentation_data(
#     detections_dataset=dataset_home,
#     output_dir='../data/masked',
#     imagenet_mean=[0.485, 0.456, 0.406]  # ImageNet RGB means
# )


#### Silhouette Masking

In [ ]:
def get_ellipse_pool(detections_dataset, confidence_threshold=0.3):
    ellipses_pool = []

    for item in tqdm(detections_dataset):
        image_path = Path("../data/images") / Path(item[0]).name
        detections = item[-1]

        keep_indices = [i for i, conf in enumerate(detections.confidence) if conf >= confidence_threshold]

        height, width = cv2.imread(str(image_path)).shape[:2]
        
        for i in keep_indices:
            x1, y1, x2, y2 = detections.xyxy[i]

            # create blank mask
            mask = np.zeros((height, width), dtype=np.uint8)

            # center, axes lengths, angle
            center = ((x1 + x2) / 2, (y1 + y2) / 2)
            axes = ((x2 - x1) / 2, (y2 - y1) / 2)
            angle = 0  # no rotation

            # draw ellipse
            mask = draw_random_pear_blob_cv(mask, center, axes[0], axes[1],
                         irregularity=0.6, spikiness=0.4,
                         vertical_stretch=1.2, bottom_bias=0.4)
            ellipses_pool.append(mask)

    ellipses_pool = np.stack(ellipses_pool, axis=0)
    return ellipses_pool

def get_silhouettes_pool(detections_dataset, confidence_threshold=0.3):
    social_actors_silhouettes=[]
    for item in tqdm(detections_dataset):
        image_path = Path("../data/images") / Path(item[0]).name
        detections = item[-1]
        
        # Load original image
        original_image = cv2.imread(str(image_path))
        original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

        keep_indices = [i for i, conf in enumerate(detections.confidence) if conf >= confidence_threshold]
        social_actors_silhouettes.append(detections.mask[keep_indices])
    social_actors_silhouettes = np.concatenate(social_actors_silhouettes, axis=0)
    return social_actors_silhouettes
        
def add_masks_to_image(detections_dataset, output_dir, masks_pool, imagenet_mean=[0.485, 0.456, 0.406], resize_to=(256, 144)):
    
    env_dir = Path(output_dir) / "environment_plus"
    env_dir.mkdir(parents=True, exist_ok=True)

    for item in tqdm(detections_dataset):
        image_path = Path("../data/masked/environment_elipses") / Path(item[0]).name
        
        # Load original image
        original_image = cv2.imread(str(image_path))
        original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
        height, width = original_image.shape[:2]
        
        # Get filename for saving
        filename = Path(image_path).name 

        idxs = random.sample(range(masks_pool.shape[0]), 15)
        filtered_masks = masks_pool[idxs]   # shape (10, 1080, 1920)
        # Combine all masks into one
        combined_mask = combine_masks(filtered_masks, height, width)
        
        # Create environment image (room only, people and robot masked out)
        env_image = apply_mask_with_mean(
            original_image, combined_mask, imagenet_mean, keep_masked=False
        )

        env_path = env_dir / filename
    
        pil_image = Image.fromarray(env_image)
        if resize_to:
            pil_image = pil_image.resize(resize_to, Image.BILINEAR)
        pil_image.save(env_path)


def draw_random_pear_blob_cv(mask, center, avg_radius_x, avg_radius_y,
                             irregularity=0.3, spikiness=0.2, num_points=14,
                             vertical_stretch=1.2, bottom_bias=0.3, color=1):
    """
    Draws a random pear-shaped blob on a NumPy mask.
    - vertical_stretch > 1.0 makes shape taller
    - bottom_bias > 0 widens the lower half
    """
    cx, cy = center
    angle = 0
    points = []
    for i in range(num_points):
        angle_step = 2 * math.pi / num_points
        angle += angle_step + random.uniform(-angle_step * irregularity,
                                             angle_step * irregularity)

        # base radii
        radius_x = avg_radius_x + random.uniform(-avg_radius_x * spikiness,
                                                 avg_radius_x * spikiness)
        radius_y = avg_radius_y + random.uniform(-avg_radius_y * (spikiness * 0.4),
                                                 avg_radius_y * (spikiness * 0.4))

        # stretch vertically
        radius_y *= vertical_stretch

        # widen bottom (when pointing down)
        if math.sin(angle) > 0:
            radius_x *= (1 + bottom_bias * math.sin(angle))

        x = cx + math.cos(angle) * radius_x
        y = cy + math.sin(angle) * radius_y
        points.append((int(x), int(y)))

    cv2.fillPoly(mask, [np.array(points, dtype=np.int32)], color)
    return mask

def process_segmentation_data_eplipses(detections_dataset, output_dir, imagenet_mean=[0.485, 0.456, 0.406], confidence_threshold=0.3):
    env_dir = Path(output_dir) / "environment_elipses"
    env_dir.mkdir(parents=True, exist_ok=True)
    
    for item in tqdm(detections_dataset):
        image_path = Path("../data/masked/environment") / Path(item[0]).name
        detections = item[-1]
        
        # Load original image
        original_image = cv2.imread(str(image_path))
        original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
        height, width = original_image.shape[:2]
        
        # Get filename for saving
        filename = Path(image_path).name

        keep_indices = [i for i, conf in enumerate(detections.confidence) if conf >= confidence_threshold]
        
        filtered_masks = []
        for i in keep_indices:
            x1, y1, x2, y2 = detections.xyxy[i]

            # create blank mask
            mask = np.zeros((height, width), dtype=np.uint8)

            # center, axes lengths, angle
            center = ((x1 + x2) / 2, (y1 + y2) / 2)
            axes = ((x2 - x1) / 2, (y2 - y1) / 2)
            angle = 0  # no rotation

            # draw ellipse
            # cv2.ellipse(mask, (int(center[0]), int(center[1])),
            #             (int(axes[0]), int(axes[1])), angle, 0, 360, 1, -1)
            mask = draw_random_pear_blob_cv(mask, center, axes[0], axes[1],
                         irregularity=0.6, spikiness=0.4,
                         vertical_stretch=1.2, bottom_bias=0.4)
            filtered_masks.append(mask)
        
        # Combine all masks into one
        combined_mask = combine_masks(filtered_masks, height, width)
        
        # Create environment image (room only, people and robot masked out)
        env_image = apply_mask_with_mean(
            original_image, combined_mask, imagenet_mean, keep_masked=False
        )
        env_path = env_dir / filename

        save_image(env_image, env_path)
    

In [ ]:
# for domain in ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']:
#     detections_dataset = get_dataset(hdf5_path="../data/autodistill_segmentation_dataset.hdf5", image_dir="../data/images/", category_name=domain, deduplicate_bbox=False)

#     # Cover social agents with elipses
#     process_segmentation_data_eplipses(
#         detections_dataset=detections_dataset,
#         output_dir='../data/masked',
#         imagenet_mean=[0.485, 0.456, 0.406]
#     )

#     # Get random 100 elipses/silhouettes from the dataset
#     # masks_pool=get_silhouettes_pool(random.sample(sorted(dataset_home), 100))
#     elipses_pool=get_ellipse_pool(random.sample(sorted(detections_dataset), 100))

#     # Further obfuscate the masked people with pooled elipses
#     add_masks_to_image(
#         detections_dataset=detections_dataset,
#         output_dir='../data/masked',
#         masks_pool=elipses_pool,
#         imagenet_mean=[0.485, 0.456, 0.406],
#         )

#### Bounding box Masking

In [ ]:
def get_box_pool(detections_dataset, confidence_threshold=0.3):
    boxes_pool = []
    for item in tqdm(detections_dataset):
        image_path = Path("../data/images") / Path(item[0]).name
        detections = item[-1]
        keep_indices = [i for i, conf in enumerate(detections.confidence) if conf >= confidence_threshold]
        height, width = cv2.imread(str(image_path)).shape[:2]
        for i in keep_indices:
            x1, y1, x2, y2 = detections.xyxy[i]
            mask = np.zeros((height, width), dtype=np.uint8)
            cv2.rectangle(mask, (int(x1), int(y1)), (int(x2), int(y2)), color=1, thickness=-1)
            boxes_pool.append(mask)
    boxes_pool = np.stack(boxes_pool, axis=0)
    return boxes_pool

def overlay_mask_at_random(mask_to_overlay, target_shape):
    # Put a mask from pool at a random location on the target canvas
    H, W = target_shape
    h_mask, w_mask = mask_to_overlay.shape
    if h_mask > H or w_mask > W:  # safety check
        mask_to_overlay = cv2.resize(mask_to_overlay, (min(w_mask, W), min(h_mask, H)))
        h_mask, w_mask = mask_to_overlay.shape
    max_x = W - w_mask
    max_y = H - h_mask
    x = random.randint(0, max_x)
    y = random.randint(0, max_y)
    canvas = np.zeros((H, W), dtype=np.uint8)
    canvas[y:y+h_mask, x:x+w_mask] = np.maximum(canvas[y:y+h_mask, x:x+w_mask], mask_to_overlay)
    return canvas

def process_and_obfuscate_boxes(detections_dataset, output_dir, mask_pool, imagenet_mean=[0.485, 0.456, 0.406], confidence_threshold=0.3, pool_obfuscate_N=15, resize_to=(256, 144)):
    env_dir = Path(output_dir) / "environment_boxes_plus"
    env_dir.mkdir(parents=True, exist_ok=True)

    for item in tqdm(detections_dataset):
        image_path = Path("../data/images") / Path(item[0]).name
        detections = item[-1]
        original_image = cv2.imread(str(image_path))
        original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
        height, width = original_image.shape[:2]
        filename = Path(image_path).name

        # Mask current detections (people/agents)
        curr_masks = []
        keep_indices = [i for i, conf in enumerate(detections.confidence) if conf >= confidence_threshold]
        for i in keep_indices:
            x1, y1, x2, y2 = detections.xyxy[i]
            mask = np.zeros((height, width), dtype=np.uint8)
            cv2.rectangle(mask, (int(x1), int(y1)), (int(x2), int(y2)), color=1, thickness=-1)
            curr_masks.append(mask)

        # Obfuscate with N pool masks at random locations
        obfuscate_masks = []
        pool_indices = random.sample(range(mask_pool.shape[0]), pool_obfuscate_N)
        for idx in pool_indices:
            pool_mask = mask_pool[idx]
            obfuscate_masks.append(overlay_mask_at_random(pool_mask, (height, width)))
        
        # Combine all masks together
        all_masks = curr_masks + obfuscate_masks
        combined_mask = combine_masks(all_masks, height, width)

        # Mask out agents + obfuscation in one go
        env_image = apply_mask_with_mean(
            original_image, combined_mask, imagenet_mean, keep_masked=False
        )

        # Resize, save final image
        pil_image = Image.fromarray(env_image)
        if resize_to:
            pil_image = pil_image.resize(resize_to, Image.BILINEAR)
        pil_image.save(env_dir / filename)


In [ ]:
# for domain in ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']:
#     detections_dataset = get_dataset(hdf5_path="../data/autodistill_segmentation_dataset.hdf5", image_dir="../data/images/", category_name=domain, deduplicate_bbox=True)
    
#     # Get pool (ideally use pooled detections from all domains, not just current — more variability)
#     box_pool = get_box_pool(random.sample(sorted(detections_dataset), 100))
    
#     # One-step full masking and obfuscation, with squares
#     process_and_obfuscate_boxes(
#         detections_dataset=detections_dataset,
#         output_dir='../data/masked',
#         mask_pool=box_pool,
#         imagenet_mean=[0.485, 0.456, 0.406],
#         pool_obfuscate_N=10,  # number of obfuscating masks applied at random locations
#         resize_to=(256, 144)
#     )


#### Latency tests

In [ ]:
from pathlib import Path
from collections import defaultdict
import random
import shutil

def build_sample_dataset():
    categories = ["Hallway", "BigOffice-2", "BigOffice-3", "MeetingRoom", "SmallOffice"]
    source_dir = Path('../data/images')
    target_dir = Path('../data/latency_test/raw_files')
    target_dir.mkdir(exist_ok=True, parents=True)

    images_by_cat = defaultdict(list)

    for p in source_dir.glob("*.png"):
        name = p.name

        cat = next((c for c in categories if c in name), "Home")
        images_by_cat[cat].append(p)

    samples = {
        c: random.sample(v, min(10, len(v)))
        for c, v in images_by_cat.items()
    }

    for c, imgs in samples.items():
        for img in imgs:
            shutil.copy(img, target_dir / img.name)


categories = ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']

images_by_cat = defaultdict(list)
for p in Path('../data/latency_test/raw_images').glob("*.png"):
    cat = next((c for c in categories if c in p.name), "Home")
    images_by_cat[cat].append(p.name)

In [ ]:
import sys
sys.path.append("..")
from heuristicSplitModel import DualBranchModel
from data_processing.data_utils import get_transform
import json

IMAGENET_NORM = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model_from_checkpoint(checkpoint_file_path, config, device):
    pt_file = torch.load(checkpoint_file_path)
    model = DualBranchModel(dropout_rate=config['model']['dropout_rate'], setup=config['model']['setup'], branch_norm=config['model']['branch_norm'])
    model = model.to(device)
    model.load_state_dict(pt_file['model_state_dict'])
    return model

config_path = '../checkpoints/default_b60.base_fold=0_buffer_size=60_epochs=60_seed=42_20260218_024124_config.json'
# config_path = '../checkpoints/single_b60.nomask_fold=0_buffer_size=60_epochs=60_seed=42_20260218_043141_config.json' 
with open(config_path, "r") as f:
    config = json.load(f)
checkpoint_file_path = '../checkpoints/default_b60.base_fold=0_buffer_size=60_epochs=60_seed=42_20260218_024124_domainSmallOffice.pt'
# checkpoint_file_path = '../checkpoints/single_b60.nomask_fold=0_buffer_size=60_epochs=60_seed=42_20260218_043141_domainSmallOffice.pt'
model = load_model_from_checkpoint(checkpoint_file_path, config, DEVICE)
model.eval()

your_transform = get_transform((144, 256), IMAGENET_NORM)

In [ ]:
uber_dataset = {}
for domain in ['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice']:
    detections_dataset = get_dataset(hdf5_path="../data/autodistill_segmentation_dataset.hdf5", image_dir="../data/images/", category_name=domain, deduplicate_bbox=True)
    box_pool = get_box_pool(random.sample(sorted(detections_dataset), 100))

    filtered = [
        (Path("../data/latency_test/raw_images") / Path(img).name, detections)
        for img, detections in detections_dataset
        if Path(img).name in images_by_cat[domain]
    ]

    uber_dataset[domain] = {
        "dataset": filtered,
        "box_pool": box_pool,
    }

In [ ]:
# import pickle
# with open("uber_dataset.pkl", "wb") as f:
#     pickle.dump(uber_dataset, f)

In [ ]:
# with open("uber_dataset.pkl", "rb") as f:
#     uber_dataset = pickle.load(f)

In [ ]:
"""
Latency benchmark for the semantic context decomposition module + dual-branch
network inference. Single image at a time, in-memory (no disk I/O beyond the
initial image read), CUDA-synchronized timing.

Semantic context decomposition = panoptic segmentation (base_model.predict)
+ NMS + social/environment split + box obfuscation, all in ONE timed block.
Dual-branch network inference = model forward pass, timed separately.

Assumes: model is already instantiated with weights loaded (fold=0),
box_pool is precomputed once per domain (offline, not timed).
"""

import time
import random
import cv2
import numpy as np
import torch
from pathlib import Path
from PIL import Image

N_WARMUP = 5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
CONF_THRESHOLD = 0.3
POOL_OBFUSCATE_N = 10
RESIZE_TO = (256, 144)

device = DEVICE
model.eval()


def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def decompose_scene(img_path, detections, box_pool):
    """
    Full semantic context decomposition for one image, from raw path to
    (social_image, env_boxes_image). Includes panoptic segmentation.
    Replaces base_model.predict() + filter_detections() + with_nms() +
    process_segmentation_data() + process_and_obfuscate_boxes(), all
    in-memory, no disk writes.
    """
    original_image = cv2.imread(str(img_path))
    original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    height, width = original_image.shape[:2]

    
    detections = detections.with_nms(class_agnostic=True)
    keep_indices = [i for i, conf in enumerate(detections.confidence) if conf >= CONF_THRESHOLD]

    # --- Social / environment split (from agent segmentation masks) --- #
    agent_masks = detections.mask[keep_indices]
    agent_mask_combined = combine_masks(agent_masks, height, width)
    social_image = apply_mask_with_mean(original_image, agent_mask_combined, IMAGENET_MEAN, keep_masked=True)

    # --- Box obfuscation on top of environment (agent boxes + decoy boxes) --- #
    agent_boxes = []
    for i in keep_indices:
        x1, y1, x2, y2 = detections.xyxy[i]
        box_mask = np.zeros((height, width), dtype=np.uint8)
        cv2.rectangle(box_mask, (int(x1), int(y1)), (int(x2), int(y2)), color=1, thickness=-1)
        agent_boxes.append(box_mask)

    pool_indices = random.sample(range(box_pool.shape[0]), POOL_OBFUSCATE_N)
    decoy_boxes = [overlay_mask_at_random(box_pool[i], (height, width)) for i in pool_indices]

    box_mask_combined = combine_masks(agent_boxes + decoy_boxes, height, width)
    env_boxes_image = apply_mask_with_mean(original_image, box_mask_combined, IMAGENET_MEAN, keep_masked=False)

    social_pil = Image.fromarray(social_image).resize(RESIZE_TO, Image.BILINEAR)
    env_boxes_pil = Image.fromarray(env_boxes_image).resize(RESIZE_TO, Image.BILINEAR)

    return social_pil, env_boxes_pil

segmentation_times = []
decomposition_times = []
inference_times = []
step=0
            

for domain in categories:
    # domain_paths = [p for p in Path("../data/latency_test/raw_images").glob("*.png") if get_category(p.name) == domain]

    # # One-time, offline, not timed - box pool built from raw detections on a sample of images
    # sample_paths = random.sample(domain_paths, min(100, len(domain_paths)))
    # sample_detections = [(p, base_model.predict(p).with_nms(class_agnostic=True)) for p in sample_paths]
    # box_pool = get_box_pool(sample_detections)
    box_pool = uber_dataset[domain]["box_pool"]
    for idx, (img_path, _) in enumerate(uber_dataset[domain]["dataset"]):
        sync()
        t0 = time.perf_counter()

            # --- Panoptic segmentation --- #
        detections = base_model.predict(img_path)

        sync()
        t_seg = time.perf_counter() - t0

        # --- Semantic context decomposition: segmentation + split + obfuscation --- #
        sync()
        t0 = time.perf_counter()

        social_pil, env_boxes_pil = decompose_scene(img_path, detections, box_pool)
        social_tensor = your_transform(social_pil).unsqueeze(0).to(device)
        env_tensor = your_transform(env_boxes_pil).unsqueeze(0).to(device)

        sync()
        t_decomp = time.perf_counter() - t0

        # --- Dual-branch network inference --- #
        sync()
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model(env_tensor, social_tensor)
        sync()
        t_infer = time.perf_counter() - t0

        if step >= N_WARMUP:
            segmentation_times.append(t_seg)
            decomposition_times.append(t_decomp)
            inference_times.append(t_infer)
        step += 1


def avg_ms(times):
    return sum(times) / len(times) * 1000 if times else float("nan")

print(f"Panoptic segmentation:                                {avg_ms(segmentation_times):.2f} ms  (n={len(segmentation_times)})")
print(f"Semantic context decomposition:                       {avg_ms(decomposition_times):.2f} ms  (n={len(decomposition_times)})")
print(f"Dual-branch network inference:                        {avg_ms(inference_times):.2f} ms  (n={len(inference_times)})")
print(f"Total per-decision latency:                           {avg_ms(segmentation_times) + avg_ms(decomposition_times) + avg_ms(inference_times):.2f} ms")

In [ ]:
# single
# Panoptic segmentation:                                7312.92 ms  (n=55)
# Semantic context decomposition:                       258.10 ms  (n=55)
# Dual-branch network inference:                        10.17 ms  (n=55)
# Total per-decision latency:                           7581.19 ms

In [ ]:
# single
# Panoptic segmentation:                                7500.17 ms  (n=30)
# Semantic context decomposition:                       256.60 ms  (n=30)
# Dual-branch network inference:                        10.34 ms  (n=30)
# Total per-decision latency:                           7767.11 ms

In [ ]:
# single 
# Panoptic segmentation:                                7562.22 ms  (n=55)
# Semantic context decomposition:                       367.18 ms  (n=55)
# Dual-branch network inference:                        10.67 ms  (n=55)
# Total per-decision latency:                           7940.07 ms

In [ ]:
# dual 
# Panoptic segmentation:                                7416.60 ms  (n=55)
# Semantic context decomposition:                       268.53 ms  (n=55)
# Dual-branch network inference:                        20.09 ms  (n=55)
# Total per-decision latency:                           7705.22 ms

In [ ]:
# dual
# Panoptic segmentation:                                7057.65 ms  (n=30)
# Semantic context decomposition:                       256.46 ms  (n=30)
# Dual-branch network inference:                        20.80 ms  (n=30)
# Total per-decision latency:                           7334.91 ms

In [ ]:
# dual 
# Panoptic segmentation:                                7801.92 ms  (n=55)
# Semantic context decomposition:                       323.72 ms  (n=55)
# Dual-branch network inference:                        21.07 ms  (n=55)
# Total per-decision latency:                           8146.72 ms

In [ ]:
(7334+7705)/2

In [ ]:
(7581+7767)/2